# Make ML models for prediction gte parameters

In [1]:
%pip install --upgrade openpyxl pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Parameters

In [7]:
from config import parameters as gtep

In [8]:
for alias, parameter in gtep.items():
    print(f"{alias:<15}: {parameter}")

l              : thermal_conductivity
hc             : heat_capacity
Cp             : heat_capacity_pressure
Cv             : heat_capacity_volume
k              : adiabatic_index
gc             : gas_const
eo             : excess_oxidizing
T              : static_temperature
P              : static_pressure
D              : staticdensity
TT             : total_temperature
PP             : total_pressure
DD             : total_density
m              : mass
v              : volume
a              : sound_speed
a_critical     : critical_sound_speed
c              : absolute_velocity
u              : portable_velocity
w              : relative_velocity
mf             : mass_flow
pipi           : total_pressure_ratio
pi             : static_pressure_ratio
titi           : total_temperature_ratio
ti             : static_temperature_ratio
mach           : mach_number
Nu             : nusselt_number
efficiency     : efficiency
effeff         : total_efficiency
peff           : pressure_efficie

# Generate data

In [9]:
import time
from itertools import product
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle

from thermodynamics import gas_const, heat_capacity_p
from substance import Substance

In [10]:
def generate(**kwargs):
    """Генерация"""
    static, names, values = {}, [], []
    for k, v in kwargs.items():
        if isinstance(v, (tuple, list, np.ndarray)):
            names.append(k)
            values.append(v)
        else:
            static[k] = v

    for combination in product(*values):
        result = dict(zip(names, combination))
        yield {**static, **result}


def now() -> str:
    return time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())

In [ ]:
substance_parameters = [
    *generate(
        **{
            gtep.mf: np.arange(1, 200 + 1, 5),
            gtep.TT: np.arange(250, 1050 + 1, 20),
            gtep.PP: np.arange(1, 20 + 1, 0.5) * 10**5,
        }
    )
]

with open(f"data/substance_parameters_{len(substance_parameters)}.pkl", "wb") as file:
    pickle.dump(substance_parameters, file)

len(substance_parameters)

65600

In [ ]:
with open("data/substance_parameters_65600.pkl", "rb") as file:
    substance_parameters = pickle.load(file)

len(substance_parameters)

65600

In [14]:
functions = {
    gtep.gc: lambda temperature: gas_const("air"),
    gtep.Cp: lambda temperature: heat_capacity_p("air", temperature),
}

## Air

In [17]:
def generate_air(**kwargs):
    """Генерация воздуха"""

    for combination in generate(**kwargs):
        yield Substance(
            combination["name"],
            composition=combination["composition"],
            parameters=combination["parameters"],
            functions=combination["functions"],
        )

In [ ]:
airs = [
    *generate_air(
        name="air",
        composition=({"N2": 0.78, "O2": 0.21, "Ar": 0.009, "CO2": 0.0004}),
        parameters=substance_parameters,
        functions=functions,
    )
]

with open(f"data/airs_{len(substance_parameters)}.pkl", "wb") as file:
    pickle.dump(airs, file)

len(airs)

13200

## Compressor

In [19]:
from gte import Compressor

In [20]:
data = []
for combination in generate(
    pipi=np.arange(1.1, 12.1 + 1, 1),
    effeff=np.arange(0.84, 0.94 + 0.01, 0.01),
    # leak=np.arange(0, 1 + 0.2, 0.2),
):
    for air in tqdm(airs):
        compressor = Compressor("compressor")
        # setattr(compressor, "leak", combination["leak"])
        setattr(compressor, gtep.pipi, combination["pipi"])
        setattr(compressor, gtep.effeff, combination["effeff"])

        compressor.solve(air)

        if compressor.is_real:
            data.append(compressor.summary)

100%|██████████| 13200/13200 [00:06<00:00, 2135.24it/s]


In [ ]:
with open(f"data/compressor_{len(substance_parameters)}.pkl", "wb") as file:
    pickle.dump(data, file)

In [22]:
df = pd.DataFrame(data)
df

,name,inlet_mass_flow,inlet_total_temperature,inlet_total_pressure,outlet_mass_flow,outlet_total_temperature,outlet_total_pressure,outlet_gas_const,outlet_heat_capacity_pressure,outlet_total_density,outlet_adiabatic_index,outlet_critical_sound_speed,leak,total_pressure_ratio,total_efficiency,power
0,compressor,1,250,100000.0,1,258.243268,110000.0,287.14,1001.731420,1.483440,1.401824,294.206665,0,1.1,0.84,8.257334e+03
1,compressor,1,250,200000.0,1,258.243268,220000.0,287.14,1001.731420,2.966880,1.401824,294.206665,0,1.1,0.84,8.257334e+03
2,compressor,1,250,300000.0,1,258.243268,330000.0,287.14,1001.731420,4.450320,1.401824,294.206665,0,1.1,0.84,8.257334e+03
3,compressor,1,250,400000.0,1,258.243268,440000.0,287.14,1001.731420,5.933760,1.401824,294.206665,0,1.1,0.84,8.257334e+03
4,compressor,1,250,500000.0,1,258.243268,550000.0,287.14,1001.731420,7.417200,1.401824,294.206665,0,1.1,0.84,8.257334e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1742395,compressor,191,1050,1600000.0,191,1954.275722,19360000.0,287.14,1246.846675,34.500534,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742396,compressor,191,1050,1700000.0,191,1954.275722,20570000.0,287.14,1246.846675,36.656818,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742397,compressor,191,1050,1800000.0,191,1954.275722,21780000.0,287.14,1246.846675,38.813101,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742398,compressor,191,1050,1900000.0,191,1954.275722,22990000.0,287.14,1246.846675,40.969384,1.299196,796.350164,0,12.1,0.94,2.084862e+08


# ML

In [19]:
import os
import pickle
from colorama import Fore

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [37]:
MODELS = (
    LinearRegression,
    DecisionTreeRegressor,
    # RandomForestRegressor,
)
METRICS = (mean_absolute_error, mean_squared_error)

In [21]:
os.listdir("data")

['substance_parameters_65600.pkl', 'compressor_13200.pkl', 'airs_65600.pkl']

In [22]:
with open("data/compressor_13200.pkl", "rb") as file:
    db = pickle.load(file)
len(db)

1742400

In [27]:
df = pd.DataFrame(db)
display(df)

,name,inlet_mass_flow,inlet_total_temperature,inlet_total_pressure,outlet_mass_flow,outlet_total_temperature,outlet_total_pressure,outlet_gas_const,outlet_heat_capacity_pressure,outlet_total_density,outlet_adiabatic_index,outlet_critical_sound_speed,leak,total_pressure_ratio,total_efficiency,power
0,compressor,1,250,100000.0,1,258.243268,110000.0,287.14,1001.731420,1.483440,1.401824,294.206665,0,1.1,0.84,8.257334e+03
1,compressor,1,250,200000.0,1,258.243268,220000.0,287.14,1001.731420,2.966880,1.401824,294.206665,0,1.1,0.84,8.257334e+03
2,compressor,1,250,300000.0,1,258.243268,330000.0,287.14,1001.731420,4.450320,1.401824,294.206665,0,1.1,0.84,8.257334e+03
3,compressor,1,250,400000.0,1,258.243268,440000.0,287.14,1001.731420,5.933760,1.401824,294.206665,0,1.1,0.84,8.257334e+03
4,compressor,1,250,500000.0,1,258.243268,550000.0,287.14,1001.731420,7.417200,1.401824,294.206665,0,1.1,0.84,8.257334e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1742395,compressor,191,1050,1600000.0,191,1954.275722,19360000.0,287.14,1246.846675,34.500534,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742396,compressor,191,1050,1700000.0,191,1954.275722,20570000.0,287.14,1246.846675,36.656818,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742397,compressor,191,1050,1800000.0,191,1954.275722,21780000.0,287.14,1246.846675,38.813101,1.299196,796.350164,0,12.1,0.94,2.084862e+08
1742398,compressor,191,1050,1900000.0,191,1954.275722,22990000.0,287.14,1246.846675,40.969384,1.299196,796.350164,0,12.1,0.94,2.084862e+08


## Compressor

In [35]:
features = [f"inlet_{gtep.mf}", f"inlet_{gtep.TT}", f"inlet_{gtep.PP}", gtep.pipi, gtep.effeff, gtep.power]
targets = (gtep.pipi, gtep.effeff, gtep.power, f"outlet_{gtep.TT}", f"outlet_{gtep.PP}")

In [38]:
test_size = 0.33

for target in (gtep.pipi, gtep.effeff, gtep.power):
    models = []
    for target_ in (target, f"outlet_{gtep.TT}", f"outlet_{gtep.PP}"):
        x, y = df[features], df[target_]
        x = x.drop([target], axis=1)
        print(target_, list(x.columns))

        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_size, shuffle=True)

        x_train = DictVectorizer(sparse=False).fit_transform(x_train.to_dict("records"))
        x_test = DictVectorizer(sparse=False).fit_transform(x_test.to_dict("records"))
        y_train = y_train.to_numpy()
        y_test = y_test.to_numpy()

        for Model in MODELS:
            model_name = Model.__name__
            print(f"\t{model_name}")
            model = Model()
            model.fit(x_train, y_train)
            for metric in METRICS:
                print(f"\t\t{Fore.YELLOW}{metric.__name__:<20}: {metric(y_test, model.predict(x_test)):.6f}{Fore.RESET}")

            with open(f"models/compressor_{target_}_{target}_{model_name}.pkl", "wb") as file:
                pickle.dump(model, file)

total_pressure_ratio ['inlet_mass_flow', 'inlet_total_temperature', 'inlet_total_pressure', 'total_efficiency', 'power']
	LinearRegression
		mean_absolute_error : 1.849775
		mean_squared_error  : 5.474865
	DecisionTreeRegressor
		mean_absolute_error : 0.000000
		mean_squared_error  : 0.000000
outlet_total_temperature ['inlet_mass_flow', 'inlet_total_temperature', 'inlet_total_pressure', 'total_efficiency', 'power']
	LinearRegression
		mean_absolute_error : 84.467126
		mean_squared_error  : 13317.852775
	DecisionTreeRegressor
		mean_absolute_error : 0.000000
		mean_squared_error  : 0.000000
outlet_total_pressure ['inlet_mass_flow', 'inlet_total_temperature', 'inlet_total_pressure', 'total_efficiency', 'power']
	LinearRegression
		mean_absolute_error : 2356819.714329
		mean_squared_error  : 10018946478290.431641
	DecisionTreeRegressor
		mean_absolute_error : 217940.040905
		mean_squared_error  : 274313709408.130890
total_efficiency ['inlet_mass_flow', 'inlet_total_temperature', 'inlet_to